In [1]:
%load_ext autoreload
%autoreload 2

# RAG exploration

Tests `backend/retriever.py`, `backend/generator.py`, and `backend/rag.py` against the real ChromaDB index from the ingestion notebook.

In [2]:
import sys
sys.path.append("..")

from backend import retriever, generator, rag
from pipeline import config

d:\Aanaconda\envs\ukask\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Retriever alone

In [8]:
query = "شرایط حذف و اضافه چیست؟"
results = retriever.search(query, k=5)

for r in results:
    m = r["metadata"]
    print(f"dist={r['distance']:.3f}  page={m['page']}  article={m['article_numbers']}  type={m['chunk_type']}")
    print(r["text"])
    print("---")

dist=0.439  page=2  article=5,6,7,8  type=text
و فقط برنامه‌ها ی آموزشی و درسی را که بر اساس ضوابط ابالغی وزارت و مصوب شورای عالی برنامه‌ریزی آموزشی تدوین گردیده . است اجرا نماید

ماده 5 . چگونگی و ترتیب ارائه تمامی دروس هر دوره و هر رشته با رعایت پیش‌نیاز (تقدم و تأخر) هر درس طبق برنامه درسی. مصوب بر عهده دانشگاه است تبصره 1. دانشجو در آخرین نیمسال تحصیلی که منجر به دانش آموختگی می شود، از رعایت مقررات مربوط . به این ماده معاف است ماده 6 . در صورت به دست نیاوردن نمره قبولی در خارج از سنوات مجاز تحصیلی در هر درس یا حذف ، غیرموجه درس به تشخیص دانشگاه پرداخت هزینه برای دانشجوی مشمول آموزش رایگان جهت انتخاب مجدد همان درس یا درس جایگزین آن، توسط هیأت امنای دانشگاه تصمیم‌گیری می‌شود ماده 7 . . شرکت دانشجو در تمام جلسات کلاس درس الزامی است تبصره 1. اگر دانشجو در درسی بیش از 3 16 جلسات کلاس و یا در جلسه امتحان ، پایان نیمسال همان درس غیبت کند نمره آن درس صفر ثبت می شود. در صورتی که به تشخیص شورای آموزشی دانشگاه، غیبت دانشجو موجه تشخیص داده شود، آن درس صفر و نمره آن، از مجموع درس‌ها ی آن نیمسا

### Tuning LLM Call Denial

In [4]:
test_queries = {
    "شرایط حذف و اضافه چیست؟": True,
    "حداکثر تعداد واحد مجاز در هر ترم چقدر است؟": True,
    "شرایط استفاده از سنوات تحصیلی چیست؟": True,
    "مراحل دریافت گواهی اشتغال به تحصیل چگونه است؟": True,
    "دانشجوی مشروط چه محدودیت‌هایی دارد؟": True,
    "حداقل معدل برای عدم مشروطی چند است؟": True,
    "شرایط تغییر رشته چیست؟": True,
    "مهلت حذف پزشکی درس چقدر است؟": True,
    "حداکثر تعداد ترم‌های مشروطی که منجر به اخراج می‌شود چند است؟": True,
    "نحوه اعتراض به نمره امتحان چگونه است؟": True,
    "حداکثر سنوات مجاز تحصیل در مقطع کارشناسی چند سال است؟": True,
    "شرایط استفاده از مرخصی تحصیلی چیست؟": True,
    "تعداد غیبت مجاز در هر درس چقدر است؟": True,
    "شرایط دفاع از پایان‌نامه چیست؟": True,
    "حداقل واحد برای دانش‌آموختگی چند واحد است؟": True,

    # off-topic, obviously unrelated
    "بهترین رستوران کرمان کدام است؟": False,
    "پایتخت فرانسه کجاست؟": False,
    "دستور پخت قورمه سبزی چیست؟": False,
    "آب و هوای امروز تهران چطور است؟": False,
    "بهترین فیلم‌های سال ۲۰۲۵ کدامند؟": False,
    "چگونه می‌توان وزن کم کرد؟": False,
    "قیمت دلار امروز چند است؟": False,
    "برنده جام جهانی فوتبال چه تیمی بود؟": False,
    "بهترین روش یادگیری زبان انگلیسی چیست؟": False,
    "مسافت بین تهران و کرمان چقدر است؟": False,
    "علائم سرماخوردگی چیست؟": False,

    # off-topic, but "hard negatives", academic-sounding, wrong scope
    "آیین‌نامه دانشگاه تهران برای مرخصی تحصیلی چیست؟": False,
    "شرایط استخدام هیئت علمی دانشگاه چیست؟": False,
    "دانشجویان چگونه می‌توانند وام دانشجویی دریافت کنند؟": False,
    "بهترین دانشگاه‌های ایران برای رشته کامپیوتر کدامند؟": False,
}

for q, is_relevant in test_queries.items():
    results = retriever.search(q, k=1)
    sim = 1 - results[0]["distance"]
    label = "relevant " if is_relevant else "off-topic"
    print(f"{label}  sim={sim:.3f}  {q}")

relevant   sim=0.561  شرایط حذف و اضافه چیست؟
relevant   sim=0.687  حداکثر تعداد واحد مجاز در هر ترم چقدر است؟
relevant   sim=0.640  شرایط استفاده از سنوات تحصیلی چیست؟
relevant   sim=0.598  مراحل دریافت گواهی اشتغال به تحصیل چگونه است؟
relevant   sim=0.698  دانشجوی مشروط چه محدودیت‌هایی دارد؟
relevant   sim=0.607  حداقل معدل برای عدم مشروطی چند است؟
relevant   sim=0.660  شرایط تغییر رشته چیست؟
relevant   sim=0.693  مهلت حذف پزشکی درس چقدر است؟
relevant   sim=0.680  حداکثر تعداد ترم‌های مشروطی که منجر به اخراج می‌شود چند است؟
relevant   sim=0.621  نحوه اعتراض به نمره امتحان چگونه است؟
relevant   sim=0.739  حداکثر سنوات مجاز تحصیل در مقطع کارشناسی چند سال است؟
relevant   sim=0.701  شرایط استفاده از مرخصی تحصیلی چیست؟
relevant   sim=0.662  تعداد غیبت مجاز در هر درس چقدر است؟
relevant   sim=0.718  شرایط دفاع از پایان‌نامه چیست؟
relevant   sim=0.643  حداقل واحد برای دانش‌آموختگی چند واحد است؟
off-topic  sim=0.411  بهترین رستوران کرمان کدام است؟
off-topic  sim=0.261  پایتخت فرانسه کجاست؟
of

## Step 2, generator alone

Reuse the chunks from step 1, so we're only testing the LLM call and prompt, not retrieval again.

In [9]:
answer = generator.generate_answer(query, results)
print(answer)

بر اساس اسناد ارائه‌شده، تعیین‌کننده زمان‌بندی و ترتیب انجام امور آموزشی از جمله حذف و اضافه، تقویم آموزشی دانشگاه است (سند ۴، ماده ۱۲). در خصوص شرایط خاص حذف، دانشجو می‌تواند یک هفته قبل از شروع امتحانات پایان نیمسال، با موافقت دانشگاه، صرفاً یک درس نظری را حذف کند (سند ۵، ماده ۱۸). همچنین درخواست‌های حذف ترم یا واحد درسی، تنها در صورتی قابل بررسی است که دانشجو در هیچ امتحانی شرکت نکرده و استاد نیز غیبت کلاسی ثبت ننموده باشد؛ در غیر این صورت امکان بررسی وجود ندارد (سند ۳، ماده ۷ و سند ۲، ماده ۷).


## Running all five sample questions

In [13]:
sample_questions = [
    "شرایط حذف و اضافه چیست؟",
    "حداکثر تعداد واحد مجاز در هر ترم چقدر است؟",
    "شرایط استفاده از سنوات تحصیلی چیست؟",
    "مراحل دریافت گواهی اشتغال به تحصیل چگونه است؟",
    "دانشجوی مشروط چه محدودیت‌هایی دارد؟",
]

for q in sample_questions:
    result = rag.answer_question(q, k=10)
    print("Q:", q)
    print("A:", result["answer"])
    print("=" * 60)

Q: شرایط حذف و اضافه چیست؟
A: بر اساس آیین‌نامه‌های آموزشی، تصمیم‌گیری درباره زمان حذف و اضافه، اعلام و ثبت نمره و تاریخ تجدید نظر، مطابق با تقویم آموزشی دانشگاه می‌باشد (سند ۴، صفحه ۵۱، ماده ۱۲). همچنین دانشجویان موظف‌اند در زمان‌های اعلام‌شده توسط دانشگاه برای ثبت‌نام و انتخاب واحد مراجعه کنند و عدم مراجعه بدون عذر موجه، به منزله ترک تحصیل محسوب می‌شود (سند ۴، صفحه ۲۸، ماده ۱۵ و تبصره ۳؛ سند ۶، صفحه ۷، تبصره ۳).
Q: حداکثر تعداد واحد مجاز در هر ترم چقدر است؟
A: حداکثر تعداد واحد مجاز در هر ترم تحصیلی بسته به مقطع تحصیلی متفاوت است: برای دوره کارشناسی ۲۰ واحد و در شرایط خاص (معدل حداقل ۱۷ یا آخرین ترم) تا ۲۴ واحد (سند ۱ و ۳، ماده ۱۷ تبصره ۱ و ۲)، برای دوره کارشناسی ارشد ۱۳ تا ۱۴ واحد و برای دوره دکتری ۱۰ واحد است (سند ۲، خلاصه قوانین آموزشی). همچنین اگر دانشجو حداکثر با ۲۴ واحد در یک نیمسال دانش‌آموخته شود، می‌تواند تا ۲۴ واحد اخذ کند (سند ۴، ماده ۶ تبصره ۲).
Q: شرایط استفاده از سنوات تحصیلی چیست؟
A: استفاده از سنوات تحصیلی بر اساس مقطع و ضوابط خاصی تعریف شده است؛ به‌طوری که در دوره‌ها

## Refusal check

In [14]:
off_topic = "بهترین رستوران کرمان کدام است؟"
result = rag.answer_question(off_topic, k=5)
print(result["answer"])
print()
print("matches expected refusal:", result["answer"].strip() == generator.NOT_FOUND_MESSAGE)

پاسخ این سؤال در اسناد موجود یافت نشد.

matches expected refusal: True
